# IARM-X: pretrain and chat-tune on Colab, Kaggle or any Jupyter GPU

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/godsonj64/IARM-X/blob/claude/eloquent-dirac-9cqy7j/notebooks/iarmx_colab.ipynb)

This notebook clones the repository, installs it, checks the GPU, and trains the **100M-parameter IARM-X** model in two stages:

1. **Pretraining** on FineWeb-Edu, streamed from the Hugging Face Hub (no 20 GB download), with a **2048-token context**.
2. **Chat fine-tuning** on UltraChat-200k, then a chat with the model.

**Before you start:** pick a GPU runtime (Colab: *Runtime → Change runtime type → GPU*; Kaggle: *Settings → Accelerator → GPU* and *Internet → on*). Then run the cells from top to bottom.

**Plans** (set `PLAN` in the next cell):

| `PLAN` | What it does | T4 (free Colab/Kaggle) | A100 / H100 |
|---|---|---|---|
| `"pilot"` (default) | 50M pretraining tokens + 200 chat-tuning steps. Checks that everything works and **measures this GPU's real speed**. | ~1 h | ~5–10 min |
| `"full"` | The real recipe: 10B tokens + 2 chat-tuning epochs | days: not practical | ~10–25 h (the pilot prints an estimate) |

**Disconnects are fine.** Checkpoints go to Google Drive (Colab) or the notebook's working directory. Rerun the cells and training **resumes exactly where it stopped** (`--resume auto`).

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/godsonj64/IARM-X.git"
BRANCH = "claude/eloquent-dirac-9cqy7j"   # switch to "main" once this branch is merged

PLAN = "pilot"                 # "pilot" or "full" (see the table above)
PILOT_TOKENS = 50_000_000      # pretraining tokens for the pilot
PILOT_SFT_STEPS = 200          # chat-tuning steps for the pilot
USE_DRIVE = True               # Colab: keep checkpoints in Google Drive across disconnects
COMPILE = None                 # None = torch.compile only for the full run on GPUs that support it

## 2. Check the GPU

In [ ]:
import os, sys, shutil, torch

assert torch.cuda.is_available(), "No GPU: Colab -> Runtime -> Change runtime type -> GPU; Kaggle -> Settings -> Accelerator -> GPU"
NGPU = torch.cuda.device_count()
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{NGPU} x {name}  (compute capability {cap[0]}.{cap[1]}, {mem:.0f} GB)  torch {torch.__version__}")
print("precision:", "bf16" if cap[0] >= 8 else "fp16 with loss scaling (no native bf16 on this GPU)")
if COMPILE is None:
    COMPILE = PLAN == "full" and cap[0] >= 7   # torch.compile needs compute capability >= 7.0
print("torch.compile:", COMPILE)

## 3. Clone and install

In [ ]:
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/working")
WORKDIR = "/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else os.getcwd()
REPO_DIR = os.path.join(WORKDIR, "IARM-X")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull --ff-only
else:
    # For a private repository use https://<token>@github.com/<owner>/IARM-X.git
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -e ".[dev]"

## 4. Where checkpoints go

In [ ]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = f"/content/drive/MyDrive/iarmx-runs/{PLAN}"
else:
    RUN_DIR = os.path.join(WORKDIR, "iarmx-runs", PLAN)
PRETRAIN_DIR = f"{RUN_DIR}/pretrain"
SFT_DIR = f"{RUN_DIR}/sft"
os.makedirs(PRETRAIN_DIR, exist_ok=True)
os.makedirs(SFT_DIR, exist_ok=True)
print("checkpoints ->", RUN_DIR)

# Optional: a Hugging Face token raises Hub rate limits for streaming.
# Colab: add a secret named HF_TOKEN (key icon on the left); Kaggle: Add-ons -> Secrets.
try:
    if IN_COLAB:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    elif IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("HF token:", "set" if os.environ.get("HF_TOKEN") else "not set (fine)")

## 5. Sanity checks (about 2–5 minutes)

Runs the model tests, then compares the parallel recurrent scan with the per-token reference loop **on this GPU** at a 2048-token context.

In [ ]:
!python scripts/smoke_test.py
!python -m pytest -q tests/test_scan.py tests/test_model.py
!python scripts/bench_scan.py --config configs/iarmx_100m_pretrain.yaml --seq 2048 --batch 2 --reps 2

## 6. Pretrain

Streams FineWeb-Edu and packs it into 2048-token rows (nothing is truncated). The log prints the loss, learning rate and **tokens/s**.

**If the session disconnects, rerun the notebook from the top.** Training resumes from the newest checkpoint with the exact data position.

In [ ]:
TOKENS = PILOT_TOKENS if PLAN == "pilot" else 10_000_000_000
launcher = f"torchrun --standalone --nproc_per_node={NGPU}" if NGPU > 1 else "python"
overrides = [
    f"training.output_dir={PRETRAIN_DIR}",
    f"training.target_tokens={TOKENS}",
    "training.precision=auto",
    f"training.compile={str(COMPILE).lower()}",
    "training.log_every=10",
    f"training.num_workers={min(2, os.cpu_count() or 1)}",
]
if PLAN == "pilot":
    overrides += ["training.warmup_steps=50", "training.save_every=200"]
else:
    overrides += ["training.save_every=500"]
cmd = f"{launcher} -m iarmx.training.train --config configs/iarmx_100m_pretrain.yaml --resume auto " + " ".join(f"--set {o}" for o in overrides)
print(cmd)
!{cmd} 2>&1 | tee -a {PRETRAIN_DIR}/train.log

## 7. What would the full 10B-token run cost on this GPU?

Uses the tokens/s measured above. The first log lines include `torch.compile` warm-up, so they are skipped.

In [ ]:
import re, statistics
log = open(f"{PRETRAIN_DIR}/train.log").read()
rates = [float(r.replace(",", "")) for r in re.findall(r"tok/s=([\d,]+)", log)]
if rates:
    rate = statistics.median(rates[len(rates) // 2:])
    hours = 10e9 / rate / 3600
    print(f"measured {rate:,.0f} tokens/s on {NGPU} x {name}")
    print(f"10B tokens would take about {hours:,.1f} hours here")
    for label, usd in [("RTX 4090 rental", 0.30), ("H100 interruptible", 0.90), ("H100 on-demand", 1.74)]:
        print(f"  if this GPU cost ${usd:.2f}/h ({label} prices): ${hours * usd:,.2f}")
else:
    print("no tokens/s lines yet: run the pretraining cell first")

## 8. Chat fine-tuning (UltraChat-200k)

Starts from the pretrained model and trains on conversations of up to 2048 tokens. The first run downloads and tokenizes UltraChat (about 10–20 minutes on a notebook CPU).

In [ ]:
overrides = [
    f"training.init_from={PRETRAIN_DIR}/final",
    f"training.output_dir={SFT_DIR}",
    "training.precision=auto",
    f"training.compile={str(COMPILE).lower()}",
    "training.log_every=10",
    f"training.num_workers={min(2, os.cpu_count() or 1)}",
]
if PLAN == "pilot":
    overrides += [f"training.max_steps={PILOT_SFT_STEPS}", "training.epochs=null",
                  "training.warmup_steps=20", "training.save_every=100"]
cmd = f"{launcher} -m iarmx.training.train --config configs/iarmx_100m_sft.yaml --resume auto " + " ".join(f"--set {o}" for o in overrides)
print(cmd)
!{cmd} 2>&1 | tee -a {SFT_DIR}/train.log

## 9. Chat with the model

A pilot model has seen only 50M tokens, so expect fluent-looking but mostly wrong answers. The full run is what makes it useful.

In [ ]:
import shlex
PROMPT = "Explain what photosynthesis is in two sentences."
!python -m iarmx.generate --model {SFT_DIR}/final --chat --prompt {shlex.quote(PROMPT)} --max-new-tokens 200 --temperature 0.7

## 10. Associative-recall check

In [ ]:
!python -m iarmx.eval.runner --model {SFT_DIR}/final --pairs 16

## Notes

- **Context length:** the model trains on 2048-token windows, so prompt plus answer should stay under about 2048 tokens (~1,500 words).
- **Kaggle:** `/kaggle/working` (20 GB) survives only if you *Save Version*. Otherwise rerunning starts over. Two T4s are used automatically through `torchrun`.
- **Other notebooks** (Lightning, Paperspace, RunPod, Vast Jupyter): checkpoints go to `./iarmx-runs`. Point `RUN_DIR` at persistent storage if the machine can be preempted.
- **Pre-tokenized data** (faster, and resumable across different GPU counts): on a machine with ~25 GB of free disk, run `python scripts/pretokenize.py --out data/tokens/fineweb-edu-10bt-gpt2 --num-proc 8`, then use `configs/iarmx_100m_pretrain_memmap.yaml`. See the README.
- **Costs and the math behind them:** `docs/TRAINING_BUDGET.md`.